<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/GEMS_EXHAUSTIVE_GEOLOGICAL_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEMS exhaustive geological discovery EDA (code rev gde.1-r1)

Native 100 m analysis of the GeoDAWN stack: landscape (A), mapped-fault geometry (B), fault-centred signatures against matched controls (C), scale and localisation (D), cross-modal evidence (E), label bias (F), regional generalisation (G), forensics of the frozen B0 / B1 / W2 held-out maps (H), a reviewable discovery-candidate atlas (I), an opt-in cached DEM pilot, then dashboards, findings, a feature-engineering hypothesis register and an evidence-based modeling decision.

| mode | meaning |
|---|---|
| `QUICK_LOOK` | 600 fault stations, 100 bootstrap draws, scales 0.5 / 2 / 10 km |
| `FULL_EDA` | 5000 fault stations, full bootstrap, all scales |
| `RESUME` | FULL_EDA settings, validated caches always reused |

**No model is trained.** Statistics are cluster-bootstrapped over fault systems (or regions, as stated); unmapped locations are matched controls, not confirmed non-faults. Every figure writes PNG + SVG + its data CSV + an interpretation JSON computed from its own statistics. The public leaderboard is never a tuning target. Outputs: `MyDrive/GEMS_2/outputs/geological_discovery_eda/`.

In [ ]:
CONFIG = {
    'repo': 'mybenkhadda/GEMS---drivendata',
    'ref': 'main',                         # pin the pushed commit SHA
    'drive_root': '/content/drive/MyDrive/GEMS_2',
    'local_root': '/content/gems_work',

    'mode': 'QUICK_LOOK',                  # QUICK_LOOK | FULL_EDA | RESUME
    'stages': {'inventory': True, 'global_geology': True, 'fault_geometry': True, 'fault_profiles': True,
               'modality_analysis': True, 'spatial_scales': True, 'crossmodal_evidence': True, 'label_bias': True,
               'regional_shift': True, 'model_forensics': True, 'unmapped_candidates': True,
               'dem_pilot': False, 'scientific_synthesis': True},
    'resolution': 'native', 'pixel_size_m': 100,
    'scales_km': [0.2, 0.5, 1, 2, 5, 10, 20],
    'sampling': {'seed': 2026, 'max_fault_systems': None, 'matched_controls_per_fault': 5,
                 'profile_half_width_km': 10, 'along_strike_spacing_m': 500,
                 'n_regional_examples': 48, 'n_model_error_examples': 48,
                 'control_distance_km': [2.0, 10.0], 'control_sensitivity_km': [[1.0, 10.0], [5.0, 20.0]]},
    'figures': {'enabled': True, 'format': ['png', 'svg'], 'dpi': 220, 'save_plot_data': True},
    'statistics': {'bootstrap_units': 'fault_system', 'n_bootstrap': 500, 'min_effect': 0.10,
                   'min_region_stations': 40},
    'models': {'load_frozen_B0': True, 'load_frozen_B1': True, 'load_frozen_W2': True, 'train_models': False},
    'dem': {'enabled': False, 'cached_only': True, 'pilot_only': True},   # cached mosaic / tiles only, never a full download

    'reuse_cached': True,
    'save_all_artifacts': True,
    'sync_to_drive': True,
    'run_synthetic_tests': True,
}
PRIVATE_REPO = False
EXPECT = {'code_rev': 'gde.1-r1', 'commit': None}

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
REPO_URL = 'https://github.com/' + CONFIG['repo'] + '.git'

def _git_env():
    if not PRIVATE_REPO:
        return None
    from google.colab import userdata
    hdr = base64.b64encode(('x-access-token:' + (userdata.get('GITHUB_TOKEN') or '')).encode()).decode()
    return {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
            'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + hdr}

def git(*args, auth=False, check=True):
    r = subprocess.run(['git', *args], capture_output=True, text=True, env=_git_env() if auth else None)
    if check and r.returncode:
        raise RuntimeError('git ' + ' '.join(args[:4]) + ' failed:\n' + r.stderr[-1500:])
    return r

if not Path(CODE, '.git').exists():
    git('clone', '-q', REPO_URL, CODE, auth=True)
git('-C', CODE, 'fetch', '-q', '--tags', 'origin', auth=True)
ref = CONFIG['ref']
if git('-C', CODE, 'rev-parse', '--verify', '-q', 'refs/remotes/origin/' + ref, check=False).returncode == 0:
    git('-C', CODE, 'checkout', '-q', '-B', 'eda_colab_active', 'origin/' + ref)
else:
    git('-C', CODE, 'checkout', '-q', '--detach', ref)
HEAD = git('-C', CODE, 'rev-parse', 'HEAD').stdout.strip()
if 'gems_bench' in sys.modules or 'gems_eda' in sys.modules:
    raise RuntimeError('gems packages already imported: Runtime -> Restart session, then run from the top')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', CODE + '[colab]', 'pytest', 'tabulate'], check=True)
sys.path.insert(0, CODE)
os.environ['GEMS_CODE_COMMIT'] = HEAD

import pandas as pd
from IPython.display import HTML, Image, Markdown, display
import gems_eda.discovery as GD
from gems_eda.discovery import run as GR_EDA
from gems_bench.geomoe import run as GR
from gems_bench.leaderboard_focus import factory as FA
assert GD.EDA_CODE_REV == EXPECT['code_rev'], f'code rev {GD.EDA_CODE_REV} != {EXPECT["code_rev"]}'
assert not EXPECT['commit'] or HEAD.startswith(EXPECT['commit']), f'HEAD {HEAD} != {EXPECT["commit"]}'
NEED_ENGINE = CONFIG['stages']['model_forensics'] or CONFIG['stages']['unmapped_candidates']
F = FA.Factory({**GR.factory_config({**GR.NB_DEFAULTS, **{k: CONFIG[k] for k in ('repo', 'ref', 'drive_root', 'local_root', 'reuse_cached', 'sync_to_drive')}})}, HEAD) if NEED_ENGINE else None
E = GR_EDA.ExhaustiveEDA(CONFIG, HEAD, factory=F)
OUT = E.ws.root

def show(rel, cols=None, n=40):
    p = OUT / rel
    if not p.exists():
        print('ABSENT:', rel); return None
    df = pd.read_csv(p) if p.suffix == '.csv' else pd.read_parquet(p)
    display(df[[c for c in (cols or df.columns) if c in df]].head(n)); return df

def figs(part, n=6):
    C = E.ws.load_table('figures/FIGURE_CATALOG.csv')
    if not len(C):
        print('no figure yet'); return
    C = C[C.part == part]
    display(C[['figure_id', 'title', 'status', 'summary']])
    for r in C[C.png.notna()].sort_values(['priority', 'figure_id']).head(n).itertuples():
        display(Markdown(f'**{r.figure_id}** {r.title} [{r.status}]'))
        display(Image(str(OUT / r.png), width=900))

print('SHA', HEAD, '| rev', GD.EDA_CODE_REV, '| mode', E.c['mode'], '| stations', E.c['max_fault_stations'], '| bootstrap', E.n_boot)
print('Drive outputs:', E.drive / E.out)

## [00] Synthetic tests (enforced) -> [e01] data inventory and audit of the previous EDA

A failing test (`TESTS_FAILED`) blocks every downstream stage. The inventory checks the three rasters against the recorded properties (EPSG 32611, 100 m, 3730 x 3292, 19 bands) and re-checks the defects of the previous EDA (malformed transform labels, saturated evidence).

In [ ]:
E.s00_tests()
if E.status['00_tests'] == 'TESTS_FAILED':
    print((OUT / 'audit/pytest_output.txt').read_text()[-4000:])
show('audit/selftests.csv', ['test', 'critical', 'status', 'detail'])
E.s_inventory()
show('audit/raw_data_inventory.csv')
show('audit/previous_eda_audit.csv', ['finding_id', 'classification', 'check_result', 'claim', 'check_detail'])

## [e02] Part A - geological landscape, [e03] Part B - mapped-fault geometry

Label-free physical domains (k-means on 5 km block descriptors), band characterisation, autocorrelation and effective support; fault lengths, orientations, curvature, junctions, fragmentation and network complexity at native resolution.

In [ ]:
E.s_landscape(); figs('A')
E.s_geometry(); figs('B')

## [e04] Fault stations + matched controls -> [e05] Part C - fault signatures -> [e06] Part D - scale and localisation

Controls are matched on region and physical setting (terrain slope x magnetic gradient terciles), 2-10 km from mapped traces (sensitivity bands 1-10 and 5-20 km). Effects are Cliff's deltas with a cluster bootstrap over fault systems; SUPPORTED needs the CI outside +-`min_effect`.

In [ ]:
E.s_profiles()
E.s_part('e05_signatures', 'parts_cd', 'PART_C'); figs('C')
show('tables/matched_control_tests.csv', ['signal', 'metric', 'effect', 'ci_lo', 'ci_hi', 'status', 'robustness'])
E.s_part('e06_scales', 'parts_cd', 'PART_D'); figs('D')
show('tables/scale_localization.csv')

## [e07] Part E - cross-modal evidence, [e08] Part F - label bias, [e09] Part G - regional generalisation

Independent families: magnetic, gravity, terrain, basement, conductivity (derivatives of one survey are never counted twice). The label-free lineament inventory (`datasets/structural_lineaments.parquet`) is built from 1 km crests without the fault map; mapping status is attached afterwards for analysis only.

In [ ]:
E.s_part('e07_crossmodal', 'parts_efg', 'PART_E'); figs('E')
E.s_part('e08_label_bias', 'parts_efg', 'PART_F'); figs('F')
E.s_part('e09_regional', 'parts_efg', 'PART_G'); figs('G')
show('tables/regional_robustness.csv')

## [01] Frozen engine -> [03] validation units -> [e11] Part H - model forensics

Reuses the submission factory and the GeoMoE / reset machinery: zone-stitched held-out B0 / B1 / W2 ranks and the frozen submitted W2 map. No training, no new inference. Without the engine these stages are `BLOCKED_UPSTREAM` (never a conclusion).

In [ ]:
if F is not None:
    E.s01_attach(); E.s03_scenarios()
E.s_forensics(); figs('H')
show('tables/model_failure_taxonomy.csv')

## [e12] Part I - discovery-candidate atlas and blind review

Open `atlas/atlas_index.html` on Drive. **Review EVIDENCE_ONLY panels first** (no mapped faults, no model output); DIAGNOSTIC panels reveal mapped faults, the frozen W2 map and B0 / B1 ranks. Reviews append to `atlas/reviewed_annotations.csv` (never overwritten); `reviewer` is required. Candidates are not discoveries and unreviewed candidates are not negatives.

In [ ]:
E.s_candidates(); figs('I', 3)
C = show('atlas/atlas_index.csv', ['candidate_id', 'category', 'region_id', 'review_split', 'supporting_families', 'distance_to_known_fault_km', 'alternative_explanation'], n=60)
if C is not None and len(C):
    display(Image(str(OUT / f'atlas/panels/{C.candidate_id.iloc[0]}_evidence_only.png')))

In [ ]:
from gems_bench.reset.atlas import review_widget
W = review_widget(E)
if W is not None: display(W)
# Alternative: fill atlas/review_template.csv (copy it on Drive) and run E.import_reviews('<path to the filled copy>')
# One review in code: E.record_review({'candidate_id': '...', 'interpretation': 'POSSIBLE_FAULT', 'confidence': 'MEDIUM',
#                                      'review_mode': 'EVIDENCE_ONLY', 'reviewer': 'name', 'evidence_sources': 'magnetic;gravity',
#                                      'supporting_evidence': '...', 'alternative_explanation': '...', 'data_license': 'n/a'})

## [e13] DEM pilot (opt-in, cached only) -> [e14] stage integrity -> [e15] synthesis

Dashboards D1-D10, `tables/evidence_synthesis.csv`, the feature-engineering hypothesis register, `reports/MODELING_DECISION.md` (15 questions, recommendation + invalidation criterion) and `reports/EXECUTIVE_RESEARCH_SUMMARY.md` (10-15 figures selected automatically). Re-run this cell after new reviews to refresh F096 / F097 and the reports.

In [ ]:
E.s_dem()
E.s_integrity(); E.s_synthesis()
show('audit/stage_integrity.csv', ['stage', 'actual_status', 'missing_outputs', 'failure_reason', 'scientific_conclusion_allowed'])
show('tables/feature_engineering_hypotheses.csv', ['hypothesis_id', 'category', 'register_status', 'source_statuses'])
for d in ('D10_executive',):
    p = OUT / f'figures/dashboards/{d}.png'
    if p.exists(): display(Image(str(p), width=1100))
for r in ('MODELING_DECISION', 'EXECUTIVE_RESEARCH_SUMMARY'):
    p = OUT / f'reports/{r}.md'
    if p.exists(): display(Markdown(p.read_text()))
print('figure failures:', {p.stem: len(json.loads(p.read_text())) for p in (OUT / 'manifests/figure_failures').glob('*.json')} if (OUT / 'manifests/figure_failures').exists() else {})

### One-shot run

`E.run()` executes every stage above in order (validated caches are reused; failed / blocked stages are recorded, never concluded on).

In [ ]:
# E.run()